# Assignment 2

# Corpus preparation (8 points)

## Question 1 (8 points)

Prepare for creating a distributional space by **counting and filtering** the surface co-occurrences in a symmetric ±5 word collocations span from the following corpus:

* A lemmatized version of the Reuters corpus (the choice of the lemmatizer is up to you). For this step, you might need a PoS-tagger: you are welcome to choose one yourself. In case you can't do PoS tagging on your own, you can use the following command to load the provided corpus in `data/reuters.pos` (uploaded as a `.zip` file, so first unzip it):

```python
with open("data/reuters.pos", "rb") as corpus_file:
    reuter_PoSTagged = pickle.load(corpus_file)
```

Remember to make motivated choices for the different strategies in building word vectors as described in class. Be explicit about:

1. what lemmas you want to describe (i.e., what will be your target vectors?);
2. how you want to describe them (i.e., what will be your contexts?);
3. what filtering strategy you are going to choose (i.e., what do you exclude?).

That means you should describe these things in text.

**Possible extra**

Actually create the space with these counts as values (i.e. make a vector with the counts, as in the subsequent questions where we put weighted counts in the vector).

The code starts by loading the provided corpus for PoS-tagger, then defining a function to create the distributional space and the words are lemmatized by WordNetLemmatizer(). A loop is then made to iterate over sentences in the corpus to lemmatize the words in a line and their corresponding PoS tags. Then, it creates a "context window" around each word and counts co-occurrences of the word and other words and updates the distributional space. Finally, the distribution space is returned and a distributional space is loaded with the given corpus.

The lemmas I want to describe is the lemmatized forms of words extracted from the corpus which represent the base or dictionary forms of words. The method I chose is using a context window to capture  the context in which words appear within the corpus. It also helps "filter" by disregarding words that are too far away to be considered as immediate context. 

I used ChatGPT to help me implement the context window beacause I didn't really know how to do it (https://chat.openai.com/share/2d20c253-2a9c-46c6-9260-05b2361cde3d). 

In [4]:
import pickle
from nltk.stem import WordNetLemmatizer
from collections import defaultdict

with open("/Users/giamihuynh/AUC_TMCI_2021/Assignments/assignment_2/data/reuters.pos", "rb") as corpus_file:
    reuter_PoSTagged = pickle.load(corpus_file)

# function to create distributional space
def create_distributional_space(corpus, window_size=5):
    distributional_space = defaultdict(lambda: defaultdict(int))
    lemmatizer = WordNetLemmatizer()
    for sentence in corpus:
        lemmatized_lines = [(lemmatizer.lemmatize(word.lower()), pos_tag[0]) for word, pos_tag in sentence]
        for i, (word, pos_tag) in enumerate(lemmatized_lines):
            context_window = lemmatized_lines[max(0, i - window_size):i] + lemmatized_lines[i + 1:min(len(lemmatized_lines), i + window_size + 1)]
            for context_word, _ in context_window:
                distributional_space[word][context_word] += 1
    return distributional_space

# create the distributional space
distributional_space = create_distributional_space(reuter_PoSTagged)

---

# Vector representations (45 points)

## Question 2 (15 points)

Weight the counts in the space you created for the previous question by using the following association measures on both spaces:

1. One **measure of your choice** among those available in the [nltk.BigramAssocMeasures](http://www.nltk.org/howto/metrics.html#association-measures) module.
2. The **Positive Local Mutual Information** measure (as shown in class/lab).

**Possible extra**

3. Also use the **smoothed ppmi measure** proposed by [Levy et al. (2015)](http://www.aclweb.org/anthology/Q15-1016). Recall that the authors proposed to smooth the ppmi by raising the context counts to the power of $\alpha$ (where $\alpha= 0.75$ is reported to work well). That is, if $V_c$ is the vocabulary of all the contexts in a given space and $f(c)$ is the context frequency, they proposed the following association measure:

$$PPMI_\alpha (w,c) = max \left(0, \ log_2 \left(\frac{p(w,c)}{p(w) \cdot p_\alpha(c)}\right)  \right) $$

$$where: \ \ p_\alpha(c) = \frac{f(c)^\alpha}{\sum_{c' \in V_c} f(c')^\alpha}$$

The function is defined by taking a distributional space as input and returns two weighted spaces, one using Pointwise Mutual Information (PMI) and the other using Positive Local Mutual Information (PLMI) measures. Then, the total number of word pairs in the distributional space is made by summing the lengths of context counts for each target word. To calculate the PMI and PLMI for each word pair in the distributional space, a for-loop is made to iterate through each target word and its associated context counts in the distributional space. Finally, the function is called with the distributional space as input, and the resulting PMI-weighted and PLMI-weighted spaces are stored in pmi_weighted_space and plmi_weighted_space. 

In [5]:
from nltk import BigramAssocMeasures
import math

# function to apply association measures to weight the counts in the distributional space
def apply_association_measures(distributional_space):
    pmi_weighted_space = defaultdict(lambda: defaultdict(float))
    plmi_weighted_space = defaultdict(lambda: defaultdict(float))

    # total no. of word pairs in the distributional space
    total_word_pairs = sum(len(context_counts) for context_counts in distributional_space.values())

    # calculate PMI and PLMI
    for target, context_counts in distributional_space.items():
        for context_word, count in context_counts.items():
            pmi = BigramAssocMeasures.pmi(count, (len(distributional_space[target]), len(distributional_space[context_word])), total_word_pairs)
            pmi_weighted_space[target][context_word] = pmi

            plmi = max(0, math.log2(count) - math.log2(len(distributional_space[target]) * len(distributional_space[context_word]) / total_word_pairs))
            plmi_weighted_space[target][context_word] = plmi

    return pmi_weighted_space, plmi_weighted_space

# apply measures to the distributional space
pmi_weighted_space, plmi_weighted_space = apply_association_measures(distributional_space)

## Question 3 (15 points)

Up to this point, you should have created 2 different distributional spaces (4 if you did the extras).

Use **Singular Value Decomposition** to reduce their dimensionality retaining only the first 100 dimensions. For this question, you can either re-use the SVD code from the lab, or import the SVD functions from external libraries such as [sklearn](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html) or [scipy](https://docs.scipy.org/doc/scipy/reference/generated/scipy.linalg.svd.html).

**Possible extra**

Find the 'optimal' number of dimensions to retain using the approach shown in the lab. Use a model with this dimensionality instead of 100.

The funciton starts by checking if the distributional space is empty and raises a value error if it does. Then, it extracts all unique target words from the keys of the distributional_space dictionary and all unique context words from the values of the dictionary. A matrix is initialized and is filled with the 'missing_values'. A for-loop iterates over each target word and context word, and for each combination it retrieves the co-occurrence count from distributional_space. TruncatedSVD is made with n components and fitted to the matrix to perform dimensionality reduction. The result is stored in reduced_matrix. Then, another for-loop iterates over each target word and creates a dictionary containing the context words and its corresponding reduced values. Finally, the function returns two objects: word_to_index dictionary and reduced_distributional_space dictionary. I tried to ask chatGPT to help optimize the running time but it did not work so I gave up on this part: https://chat.openai.com/share/d2bc8ec5-1525-44fb-bf1d-f162d2e2b71f. 

In [7]:
import numpy as np
from sklearn.decomposition import TruncatedSVD

def reduce_dimensionality(distributional_space, n_components=100, missing_value=0):
    if not distributional_space:
        raise ValueError("Distributional space is empty")

    target_words = list(distributional_space.keys())
    context_words = set(word for context_counts in distributional_space.values() for word in context_counts.keys())
    word_to_index, reduced_distributional_space = reduce_dimensionality(distributional_space)
    matrix = np.full((len(target_words), len(context_words)), missing_value)    
    
    for i, target in enumerate(target_words):
        for j, context_word in enumerate(context_words):
            matrix[i, j] = distributional_space[target].get(context_word, missing_value)
    svd = TruncatedSVD(n_components=n_components)
    reduced_matrix = svd.fit_transform(matrix)
    
    reduced_distributional_space = {}
    for i, target in enumerate(target_words):
        reduced_distributional_space[target] = {context_word: reduced_matrix[i, j] for j, context_word in enumerate(context_words)}
    return word_to_index, reduced_distributional_space

# Assuming distributional_space is defined somewhere in your code
word_to_index, reduced_distributional_space = reduce_dimensionality(distributional_space)


## Question 4 (15 points)

Train a Word2Vec model the same corpus, for example using [gensim](https://radimrehurek.com/gensim). Make sure to motivate the choice of your hyperparameters.

**Possible extra**

*Hyperparameter tuning* is the process of finding the optimal hyperparameters in a machine learning task for a given task and data set. Try performing some kind of hyperparameter tuning on your Word2Vec model. Hint: Do this after question 5, so that you'll have a way to know what hyperparameter combination is best.

---

# Evaluating on semantic similarity (17 points)

## Question 5 (17 points)

Evaluate the performance of your models on a **semantic similarity task**. Using `SimLex-999` as gold standard. Evaluate all of your models on the dataset in `data/SimLex-999.txt`, and determine the best performing model. Note: There should be 5 to 8 model evaluations in total. 5 if you did not do any extra (2 from Q2 + 2 from Q3 + 1 from Q4), and 8 if you did them all (1 from Q1, 3 from Q2 + 3 from Q3 + 2 from Q4).

1. Your evaluation should follow the approach shown in lab 4 (Section 1.6: "Evaluating your Model"), using a **correlation measure** on model predictions and the (human) gold standard. 
2. Remember to **visualize** your results (e.g., as bar plots).
3. Take note (and report) the overlap between your models and the SimLex-999 dataset, i.e., how many pairs are shared by your model and the evaluation dataset.
4. Make sure to discuss your results and provide your reasoning on them.

### Remarks

- The 'SimLex-999' dataset is described in `data/SimLex-999.README.txt`, and [the author's github page](https://fh295.github.io/simlex.html). Hint: the relevant judgements are those in the `SimLex999` column.
- To directly compare the models against the gold standard, you will have to find the *overlap* between them, i.e. the pairs that occur in your model *and* the evaluation dataset.

The function starts by reading the SimLex-999 and and extracting word pairs along with their similarity scores. It skips the header line and extract word pairs and similarity scores from each line. The similarity score function is a for-loop that iterates over each word pair and checks if both words are present in the model. If present, it retrieves the similarity score from the model; otherwise, it assumes a similarity score of 0. The word pairs and their gold similarity scores are stored in simlex_word_pairs and simlex_gold_scores. A for-loop iterates over each model and its corresponding name, and calculates predicted similarity scores. After evaluating all models, the code plots the Spearman's rank correlation coefficients (rhos) for each model using matplotlib. 

Note: It really did work before and I was able to print out a bar graph like the one in the lab but because of question 3, there are difficulties getting question 5 to run. I got PMI and PMLI have the same value, and RDS was NaN. 

In [ ]:
import scipy.stats
import matplotlib.pyplot as plt
import numpy as np

# function to read SimLex-999 dataset
def read_simlex(file_path):
    word_pairs = []
    similarity_scores = []
    with open(file_path, 'r') as file:
        next(file)  # Skip header line
        for line in file:
            parts = line.strip().split('\t')
            word1, word2, _, similarity_score, *_ = parts
            word_pairs.append((word1, word2))
            similarity_scores.append(float(similarity_score))
    return word_pairs, similarity_scores

# calculate similarity scores
def calculate_similarity_scores(model, word_pairs):
    predicted_scores = []
    for word1, word2 in word_pairs:
        if word1 in model and word2 in model:
            similarity_score = model[word1][word2] 
            predicted_scores.append(similarity_score)
        else:
            predicted_scores.append(0)  # if word pair not found, assume similarity score is 0
    return predicted_scores

# Read SimLex-999 dataset
simlex_word_pairs, simlex_gold_scores = read_simlex("/Users/giamihuynh/AUC_TMCI_2021/Assignments/assignment_2/data/SimLex-999.txt")
models = [pmi_weighted_space, plmi_weighted_space, reduced_distributional_space]
model_names = ["PMI Weighted Space", "PLMI Weighted Space", "Reduced Distributional Space"]

# evaluate each model
rhos = []
for model, model_name in zip(models, model_names):
    predicted_scores = calculate_similarity_scores(model, simlex_word_pairs)
    
    # check if predicted_scores or simlex_gold_scores are constant arrays
    if np.all(np.diff(predicted_scores) == 0) or np.all(np.diff(simlex_gold_scores) == 0):
        rho = np.nan  # assign NaN if one of the arrays is constant
    else:
        rho, _ = scipy.stats.spearmanr(predicted_scores, simlex_gold_scores)
    rhos.append(rho)
    print(f"{model_name}: {rho}")

# plotting the results
plt.bar(range(len(model_names)), rhos, align='center')
plt.xticks(range(len(model_names)), model_names, rotation=45)
plt.ylabel('Spearman\'s rho')
plt.show()


NameError: name 'reduced_distributional_space' is not defined

---